# Build a Self-Improving Multi-Agent AI System with Reflection Pattern

<a target="_blank" href="https://colab.research.google.com/github/unionai/workshops/blob/main/tutorials/multi-agent-workflows/tutorial_reflection_agent.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Learn to build a reflection agent system that iteratively improves outputs through self-critique and refinement.

**Pattern:** Generate → Critique → Refine loop until quality threshold met

```
User: "Calculate factorial of 5 and explain the result"
  ↓
Iteration 1:
  Generate: "5! = 120"
  Reflect: "Score: 5/10 - Missing explanation, no step-by-step"
  Refine: "5! = 5×4×3×2×1 = 120. This means..."
  ↓
Iteration 2:
  Reflect: "Score: 9/10 - Excellent explanation, meets threshold!"
  ✅ Done
```

**Key concepts:** Self-critique, iterative refinement, quality assessment, convergence criteria.

**vs Other Patterns:**
- **Planner**: Upfront planning → parallel execution (maximize efficiency)
- **ReAct**: Adaptive actions based on observations (maximize flexibility)
- **Reflection**: Iterative self-improvement (maximize quality)

---

## Setup

Project structure: `agents/`, `tools/`, `workflows/`, `utils/`

**Configuration:** Shared Flyte environment with Docker image + secrets. Agents inherit this config but can override for custom resources.

In [2]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone https://github.com/unionai/workshops
    %cd workshops/tutorials/multi-agent-workflows/
    !uv pip install -r requirements.txt
    
# this is just for viewing the code files within the notebook
from utils.file_viewer import view_file

In [ ]:
view_file("requirements.txt")

In [ ]:
view_file("config.py")

## Connect to Flyte Cluster
You can skip this step if you're not using a Flyte cluster and only want to run the examples locally.

Flyte gives you...

- If you don't have a Flyte cluster you can request demo access by filling out the form [here](https://flyte.org/).
- If you already have a Flyte cluster, you can connect to it by setting your endpoint in the Flyte configuration.


In [ ]:
!flyte create config \
    --endpoint tryv2.hosted.unionai.cloud \
    --auth-type headless\
    --builder remote \
    --domain development \
    --project flytesnacks

You can now adjust the configuration by modifying the `.flyte/config.yaml` file.

In [ ]:
view_file(".flyte/config.yaml")

## Set your API Key(s)

The project is setup to read in secrets from a `.env` file.

You can create this file in the root of this tutorial `tutorials/multi-agent-workflows` and add your API keys there.

But if you prefer to just enter a key once in this notebook you can run the cell below:

In [ ]:
# Skip if API key is already set in .env or environment
import os
from getpass import getpass

os.environ['OPENAI_API_KEY'] = getpass('OPENAI_API_KEY: ')

To run on the remote Flyte cluster, add the API keys as secrets. 

You can skip this step if you're not using a Flyte cluster and only want to run the examples locally.


In [ ]:
# run this and enter your API key as the input
!flyte create secret OPENAI_API_KEY

## Run the Agent

At this point you should be setup to run the reflection agent. 

I suggest giving it a try before we walk through the code in the next section.

**Run locally:**

In [7]:
!python -m workflows.reflection --request "Calculate the factorial of 5 and explain what it means" --local --quality-threshold 8 --max-iterations 3

Running workflow LOCALLY with flyte.init()

=== Reflection Multi-Agent Workflow ===
Task: Calculate the factorial of 5 and explain what it means
Quality threshold: 8/10
Max iterations: 3

[5edacccd-b2de-4175-a410-55bd0ef45bdd][4nk71bt5ingda2kvrxiigmils] ================================================================================
[5edacccd-b2de-4175-a410-55bd0ef45bdd][4nk71bt5ingda2kvrxiigmils] REFLECTION WORKFLOW - Task: Calculate the factorial of 5 and explain what it means
[5edacccd-b2de-4175-a410-55bd0ef45bdd][4nk71bt5ingda2kvrxiigmils] Quality threshold: 8/10, Max iterations: 3
[5edacccd-b2de-4175-a410-55bd0ef45bdd][4nk71bt5ingda2kvrxiigmils] ================================================================================
[5edacccd-b2de-4175-a410-55bd0ef45bdd][4nk71bt5ingda2kvrxiigmils] 
[Reflection] Step 1: Selecting appropriate agent...
[5edacccd-b2de-4175-a410-55bd0ef45bdd][4nk71bt5ingda2kvrxiigmils] [Reflection] Selected agent: math
[5edacccd-b2de-4175-a410-55bd0ef45bdd][4n

**Run on the remote Flyte cluster:**

The first time running the agent a container image will be built and pushed to the Flyte cluster.

This may take some time depending on the size of your dependencies.

In [8]:
!python -m workflows.reflection --request "Calculate the factorial of 5 and explain what it means" --quality-threshold 8 --max-iterations 3

Running workflow REMOTELY with flyte.init_from_config()

=== Reflection Multi-Agent Workflow ===
Task: Calculate the factorial of 5 and explain what it means
Quality threshold: 8/10
Max iterations: 3

16:42:23.855576 WARNING  remote_builder.py:95 -  Image                          
                         356633062068.dkr.ecr.us-east-2.amazonaws.com/union/demo
                         :flyte-a6e80122ad3ee2d24341681f57f3b7a8 found. Skip    
                         building.                                              
16:42:23.857067 WARNING  _deploy.py:376 -  Built Image for environment base_env,
                         image:                                                 
                         356633062068.dkr.ecr.us-east-2.amazonaws.com/union/demo
                         :flyte-a6e80122ad3ee2d24341681f57f3b7a8                

Execution: r9qgv65z58d2zpdfc88m
URL: https://demo.hosted.unionai.cloud/v2/domain/development/project/flytesnacks/runs/r9qgv65z58d2zpdfc88m



# Code Walkthrough

Let's walk through the code to understand how the planner agent is structured and how it works.

We'll cover all the key file types, but you can explore all the agents and tools in their respective folders.

## Infrastructure

**Decorators** - Registration system for agents and tools. 

This allows for easy addition and management of new agents and tools within the workflow.

In [ ]:
view_file("agents/planner_agent.py")

---

## Orchestrator - The Agentic Workflow

Executes plans with dependency-aware parallelism.

**Flow:**
1. Get plan from planner
2. Loop: Find steps with satisfied dependencies → Execute in parallel → Mark complete
3. For dependent steps: Inject previous results via `build_task_with_context()`

**Context passing:**
```python
# Step 2 depends on steps 0 and 1
task = """
RESULTS FROM PREVIOUS STEPS:
  - Step 0 (math): 5
  - Step 1 (math): 11

YOUR TASK:
Add the results
"""
```

**Key features:** Automatic parallelization (`asyncio.gather`), result propagation, circular dependency detection.

In [ ]:

view_file("workflows/planner.py")

## Reflection Orchestrator - Self-Improvement Loop

The reflection pattern uses iterative self-critique to improve output quality.

**Flow:**
1. **Agent Selection:** LLM picks appropriate specialist agent
2. **Initial Generation:** Agent produces first response
3. **Reflection Loop:**
   - **Critique:** Evaluate quality (1-10), identify issues, suggest improvements
   - **Check Threshold:** If score ≥ threshold → Done ✅
   - **Refine:** Generate improved response addressing critique
   - Repeat until converged or max iterations

**Quality Assessment:**
```python
reflection_data = {
  "quality_score": 6,  # 1-10 scale
  "issues": [
    "Missing step-by-step explanation",
    "No context about what factorial means"
  ],
  "suggestions": "Add detailed calculation steps and explain the concept"
}
```

**Key features:**
- Configurable quality threshold (default: 8/10)
- Tracks all iterations with issues + improvements
- Convergence detection (stops when threshold met)
- Max iterations safeguard

**Agent routing:** Uses `agent_registry` for dynamic dispatch (same as planner/ReAct).

In [3]:
view_file("workflows/reflection.py")

---

## Running the Workflow

**Local (development):**
```bash
python -m workflows.reflection --local \
  --request "your task" \
  --quality-threshold 8 \
  --max-iterations 5
```
In-process execution, fast iteration.

**Remote (production):**
```bash
python -m workflows.reflection \
  --request "your task" \
  --quality-threshold 9 \
  --max-iterations 5
```
Distributed Flyte cluster, scalable and observable.

**Parameters:**
- `--quality-threshold`: Minimum score (1-10) to accept. Higher = stricter. Default: 8
- `--max-iterations`: Max refinement cycles. Prevents infinite loops. Default: 5

**Try these:**
- Explanations: `"Calculate 5 factorial and explain what it means"`
- Complex tasks: `"Search for France's GDP and write a 2-sentence summary"`
- Code quality: `"Write Python code to find prime numbers up to 100"`
- High standards: Use `--quality-threshold 9` for very strict requirements

In [ ]:
!python -m workflows.reflection --request "Calculate 5 factorial and explain what it means" --local --quality-threshold 8 --max-iterations 3

In [ ]:
!python -m workflows.reflection --request "Count the words in 'Hello beautiful world' and explain the result" --local --quality-threshold 7 --max-iterations 2

---

## Key Takeaways

**Reflection Pattern:**
- **Self-improving:** Critiques and refines its own outputs
- **Quality-focused:** Iterates until quality threshold met
- **Structured critique:** Scores + specific issues + actionable suggestions
- **Convergence:** Stops when satisfactory or max iterations reached

**When to use each pattern:**

| Pattern | Best For | Trade-off |
|---------|----------|----------|
| **Planner** | Clear task decomposition, maximize parallelism | Fixed plan, can't adapt mid-execution |
| **ReAct** | Exploratory tasks, unknown step count | Sequential only, slower for parallelizable tasks |
| **Reflection** | High quality requirements, explanations, writing | More LLM calls, slower but higher quality |

**Combine patterns:**
- Use **Planner** to decompose, **Reflection** on final aggregation step
- Use **ReAct** with reflection at each step for ultra-high quality

**Architecture benefits:**
- Same agents/tools work with all patterns
- Type-safe, observable, scalable with Flyte
- Easy to implement new orchestration patterns

**What makes this powerful:**
The system doesn't just execute - it evaluates, critiques, and improves its outputs, leading to production-quality results.

**Next steps:**
1. Experiment with different quality thresholds
2. Compare same task across all three patterns
3. Try combining patterns (e.g., ReAct with reflection)
4. Track quality scores across iterations in logs

---

## Resources

- Full code: `tutorials/multi-agent-workflows/`
- Planner tutorial: [tutorial_planner_agent.ipynb](tutorial_planner_agent.ipynb)
- ReAct tutorial: [tutorial_react_agent.ipynb](tutorial_react_agent.ipynb)
- Flyte docs: https://docs.flyte.org
- Reflection paper: ["Reflexion: Language Agents with Verbal Reinforcement Learning"](https://arxiv.org/abs/2303.11366)
- Questions? Join the Flyte community Slack!